# AutoIntake — Bayesian Optimization on Colab (GPU)

This notebook runs the full ABEP intake optimization pipeline on Google Colab,
using GPU acceleration via CuPy for the view-factor and linear solver computations.

**What it does:**
1. Clones the repo and installs dependencies
2. Mounts Google Drive for persistent results
3. Runs Bayesian Optimization (scikit-optimize) to find optimal intake geometry
4. Saves results + generates convergence plots

**Runtime:** Select `Runtime > Change runtime type > T4 GPU` (or A100 if available).

## 1. Environment Setup

In [ ]:
# Verify GPU is available
!nvidia-smi

In [ ]:
# Mount Google Drive for persistent storage
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_RESULTS = '/content/drive/MyDrive/AutoIntake/results'
os.makedirs(DRIVE_RESULTS, exist_ok=True)
print(f'Results will be saved to: {DRIVE_RESULTS}')

In [ ]:
# Clone the repository (dataset branch has the intake_optimization package)
REPO_DIR = '/content/AutoIntake'
BRANCH = 'dataset'

if not os.path.isdir(REPO_DIR):
    !git clone -b {BRANCH} https://github.com/mattiademartino/AutoIntake.git {REPO_DIR}
else:
    !git -C {REPO_DIR} fetch origin
    !git -C {REPO_DIR} checkout {BRANCH}
    !git -C {REPO_DIR} pull origin {BRANCH}

os.chdir(REPO_DIR)
print(f'Working directory: {os.getcwd()}')
print(f'Branch: {BRANCH}')

In [ ]:
# Install dependencies
!pip install -q numpy-stl trimesh scipy matplotlib pandas scikit-optimize

# Install CuPy for GPU acceleration (matches Colab CUDA version)
import subprocess
cuda_version = subprocess.check_output(
    "nvcc --version | grep release | sed 's/.*release //' | sed 's/,.*//' | tr -d '.'",
    shell=True
).decode().strip()
print(f'Detected CUDA version code: {cuda_version}')

# CuPy wheel for the detected CUDA version
!pip install -q cupy-cuda12x 2>/dev/null || pip install -q cupy-cuda11x 2>/dev/null || echo 'CuPy install failed, will use NumPy CPU fallback'

# Verify CuPy
try:
    import cupy as cp
    print(f'CuPy {cp.__version__} — GPU: {cp.cuda.runtime.getDeviceProperties(0)["name"].decode()}')
except Exception as e:
    print(f'CuPy not available ({e}), will use NumPy CPU fallback')

## 2. Verify Setup

In [ ]:
import sys
sys.path.insert(0, REPO_DIR)

from intake_optimization.backend import has_gpu, backend_name
from intake_optimization import config

print(f'Compute backend: {backend_name()}')
print(f'GPU available:   {has_gpu()}')
print(f'On Colab:        {config.ON_COLAB}')
print(f'Repo root:       {config.REPO_ROOT}')
print(f'Results dir:     {config.RESULTS_BASE_DIR}')
print(f'HC STL exists:   {os.path.isfile(config.HC_STL_PATH)}')

## 3. Quick Smoke Test

Run a single evaluation at the baseline parameters `(a=0, d=0, e=0)` to verify the full pipeline works.

In [ ]:
import time
from intake_optimization.mesh_generator import compute_effective_L
from intake_optimization.simulation import SimulationRunner

L = compute_effective_L()
print(f'Effective channel length: L = {L:.4f} mm')

runner = SimulationRunner(base_dir='/content/smoke_test', L=L, verbose=True)
runner.setup()

t0 = time.time()
flux, feasible = runner.evaluate([0.0, 0.0, 0.0])
dt = time.time() - t0

print(f'\nBaseline result:')
print(f'  Flux     = {flux:.6e}')
print(f'  Feasible = {feasible}')
print(f'  Time     = {dt:.2f} s')
print(f'\nPipeline is working!')

## 4. Run Bayesian Optimization

Configure the optimization budget and launch.  Results are saved incrementally to Google Drive, so you can interrupt and resume.

In [ ]:
# ── Configuration ──
MAX_EVALUATIONS = 60       # Total budget
N_INITIAL_POINTS = 10      # Random exploration phase

# Output directory (on Google Drive for persistence)
from datetime import datetime
RUN_NAME = datetime.now().strftime('%Y%m%d_%H%M%S')
OUTPUT_DIR = os.path.join(DRIVE_RESULTS, RUN_NAME)

# To RESUME a previous run, set OUTPUT_DIR to that run's directory:
# OUTPUT_DIR = '/content/drive/MyDrive/AutoIntake/results/20260312_150000'

print(f'Output directory: {OUTPUT_DIR}')
print(f'Budget: {MAX_EVALUATIONS} evaluations ({N_INITIAL_POINTS} random + {MAX_EVALUATIONS - N_INITIAL_POINTS} BO)')

In [ ]:
from intake_optimization.config import RunConfig
from intake_optimization.optimizer import run_optimization

cfg = RunConfig(
    max_evaluations=MAX_EVALUATIONS,
    n_initial_points=N_INITIAL_POINTS,
    output_dir=OUTPUT_DIR,
)

run_optimization(cfg)

## 5. Visualize Results

In [ ]:
from intake_optimization.utils import HistoryLogger
from intake_optimization.visualize import (
    plot_convergence,
    plot_param_trajectories,
    plot_best_geometry,
    plot_gp_slices,
)
import matplotlib.pyplot as plt

csv_path = os.path.join(OUTPUT_DIR, 'history.csv')
logger = HistoryLogger(csv_path)
print(f'Loaded {logger.n_evaluations} evaluations')

best = logger.best
if best:
    print(f'Best flux: {best.objective:.6e}')
    print(f'Best params: a={best.params[0]:.4f}, d={best.params[1]:.4f}, e={best.params[2]:.4f}')

plots_dir = os.path.join(OUTPUT_DIR, 'plots')
os.makedirs(plots_dir, exist_ok=True)

In [ ]:
# Convergence plot
plot_convergence(logger, plots_dir)

from IPython.display import Image, display
display(Image(filename=os.path.join(plots_dir, 'convergence.png')))

In [ ]:
# Parameter trajectories
plot_param_trajectories(logger, plots_dir)
display(Image(filename=os.path.join(plots_dir, 'param_trajectories.png')))

In [ ]:
# Best geometry
plot_best_geometry(logger, plots_dir)
display(Image(filename=os.path.join(plots_dir, 'best_geometry.png')))

In [ ]:
# GP surrogate slices
plot_gp_slices(logger, plots_dir)
display(Image(filename=os.path.join(plots_dir, 'gp_slices.png')))

## 6. Export Best Result

In [ ]:
import json

if best:
    result = {
        'best_params': {'a': best.params[0], 'd': best.params[1], 'e': best.params[2]},
        'best_flux': best.objective,
        'iteration': best.iteration,
        'total_evaluations': logger.n_evaluations,
        'L_effective_mm': L,
    }
    result_path = os.path.join(OUTPUT_DIR, 'best_result.json')
    with open(result_path, 'w') as f:
        json.dump(result, f, indent=2)
    print(f'Best result saved to {result_path}')
    print(json.dumps(result, indent=2))
else:
    print('No feasible evaluation found.')